# A 組｜Week 1 關鍵數字手動驗算

直接讀主要資料集，重新計算詐欺比例、`fraudStep` 交叉表與小額交易帳戶規則；所有結果均可重跑。

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / 'src'))
from aml_insight.data.loader import load_dataset
import pandas as pd
ds = load_dataset('20K_fanin200cycle200')
nodes, tx = ds.nodes, ds.transactions
print(f'fraud accounts: {int(nodes.isFraud.sum()):,} / {len(nodes):,} = {nodes.isFraud.mean():.2%}')

fraud accounts: 1,804 / 20,000 = 9.02%


## `fraudStep` 與標籤交叉表

`fraudStep` 只作為診斷欄位，不可作為模型特徵。

In [2]:
pd.crosstab(nodes.isFraud, nodes.fraudStep.eq(-1), rownames=['isFraud'], colnames=['fraudStep == -1'])

fraudStep == -1,False,True
isFraud,,
0,0,18196
1,898,906


## 金額捷徑驗算

以非「兩端皆詐欺帳戶」交易的最小金額作為資料中實際觀察到的分界；檢查小額交易與端點標籤的關係。

In [3]:
labels = nodes.set_index('nodeid').isFraud
both = tx.sourceNodeId.map(labels).eq(1) & tx.targetNodeId.map(labels).eq(1)
threshold = tx.loc[~both, 'value'].min()
small = tx.loc[tx.value < threshold]
accounts = set(small.sourceNodeId) | set(small.targetNodeId)
flag = nodes.nodeid.isin(accounts)
precision = (flag & nodes.isFraud.eq(1)).sum() / max(flag.sum(), 1)
recall = (flag & nodes.isFraud.eq(1)).sum() / nodes.isFraud.sum()
print(f'threshold={threshold:.2f}; small transactions={len(small):,}; both endpoints fraud={both.loc[small.index].mean():.2%}')
print(f'account rule: flagged={flag.sum():,}; precision={precision:.2%}; recall={recall:.2%}')

threshold=100.04; small transactions=3,695; both endpoints fraud=100.00%
account rule: flagged=1,228; precision=100.00%; recall=68.07%
